# CELL 1: Chuẩn hóa kích thước 600x600 chuẩn JPG

In [1]:
import os
import cv2
import glob

# Danh sách các thư mục cần chuẩn hóa
FOLDERS = [r".\Face_Database", r".\Face_Test"]

def resize_and_convert_to_jpg(image_path, target_size=(600, 600)):
    img = cv2.imread(image_path)
    if img is None:
        print(f"⚠️ Không thể đọc ảnh: {image_path}")
        return None
    
    h, w = img.shape[:2]
    scale = min(target_size[0] / w, target_size[1] / h)
    nw, nh = int(w * scale), int(h * scale)
    
    interpolation = cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC
    resized_img = cv2.resize(img, (nw, nh), interpolation=interpolation)
    
    # Thêm viền đen 600x600 giữ tỉ lệ mặt
    padded_img = cv2.copyMakeBorder(
        resized_img, 
        top=(target_size[1] - nh) // 2, 
        bottom=target_size[1] - nh - (target_size[1] - nh) // 2,
        left=(target_size[0] - nw) // 2, 
        right=target_size[0] - nw - (target_size[0] - nw) // 2,
        borderType=cv2.BORDER_CONSTANT, 
        value=[0, 0, 0]
    )
    
    base_name = os.path.splitext(image_path)[0]
    new_jpg_path = f"{base_name}.jpg"
    cv2.imwrite(new_jpg_path, padded_img, [int(cv2.IMWRITE_JPEG_QUALITY), 95])
    
    if image_path.lower().endswith('.png') and os.path.exists(new_jpg_path):
        os.remove(image_path)
        
    return new_jpg_path

supported_exts = ('.jpg', '.jpeg', '.png')
for folder in FOLDERS:
    if not os.path.exists(folder):
        os.makedirs(folder)
        continue
    all_files = glob.glob(os.path.join(folder, "*.*"))
    image_files = [f for f in all_files if f.lower().endswith(supported_exts)]
    
    print(f"🔄 Đang chuẩn hóa {len(image_files)} ảnh trong thư mục '{folder}'...")
    for img_p in image_files:
        resize_and_convert_to_jpg(img_p)

print("✅ Hoàn tất chuẩn hóa ảnh về 600x600 chuẩn JPG cho cả 2 thư mục!")

🔄 Đang chuẩn hóa 18 ảnh trong thư mục '.\Face_Database'...
🔄 Đang chuẩn hóa 75 ảnh trong thư mục '.\Face_Test'...
✅ Hoàn tất chuẩn hóa ảnh về 600x600 chuẩn JPG cho cả 2 thư mục!


# CELL 2: Tạo test_label.csv chứa image_path và true_name cho tập Test

In [2]:
import os
import glob
import pandas as pd

TEST_DIR = r".\Face_Test"
CSV_PATH = "test_label.csv"

# Ánh xạ tiền tố file thành tên người nhà
NAME_MAP = {
    "dat": "Đạt",
    "quan": "Quân",
    "tien": "Tiến",
    "vu": "Vũ"
}

jpg_files = glob.glob(os.path.join(TEST_DIR, "*.jpg"))

data = []
for file_path in jpg_files:
    filename = os.path.basename(file_path)
    prefix = filename.split('_')[0].lower()
    true_name = NAME_MAP.get(prefix, "Người lạ")
    
    data.append({
        "image_path": file_path,
        "true_name": true_name
    })

df = pd.DataFrame(data)
df.to_csv(CSV_PATH, index=False, encoding='utf-8-sig')
print(f"✅ Đã tạo file '{CSV_PATH}' chứa {len(df)} mẫu thử nghiệm thực tế!")

✅ Đã tạo file 'test_label.csv' chứa 75 mẫu thử nghiệm thực tế!


# CELL 3: Chạy Mô Phỏng Vận Hành Thực Tế & Đánh Giá Chỉ Số

In [ ]:
import os
import glob
import time
import numpy as np
import pandas as pd
from deepface import DeepFace
from scipy.spatial.distance import cosine
from sklearn.metrics import balanced_accuracy_score

# --- CẤU HÌNH THƯ MỤC VÀ NHÃN ---
DB_DIR = r".\Face_Database"
TEST_DIR = r".\Face_Test"
CSV_RESULT_PATH = "benchmark_summary_optimal.csv"

NAME_MAP = {
    "dat": "Đạt",
    "quan": "Quân",
    "tien": "Tiến",
    "vu": "Vũ"
}

DETECTORS = ["retinaface", "mtcnn", "ssd"]
MODELS = ["Facenet", "ArcFace", "Facenet512", "SFace"]

# Dải ngưỡng để quét (từ 0.10 đến 0.90, mỗi bước 0.02)
THRESH_RANGE = np.arange(0.10, 0.91, 0.02)

# 1. Tải danh sách file test
jpg_test_files = glob.glob(os.path.join(TEST_DIR, "*.jpg"))
test_samples = []
for file_path in jpg_test_files:
    filename = os.path.basename(file_path)
    prefix = filename.split('_')[0].lower()
    true_name = NAME_MAP.get(prefix, "Người lạ")
    test_samples.append({
        "image_path": file_path,
        "true_name": true_name
    })

db_files = glob.glob(os.path.join(DB_DIR, "*.jpg"))
final_summary = []

print(f"🚀 Bắt đầu quét dải ngưỡng cho {len(DETECTORS) * len(MODELS)} tổ hợp mô hình...\n")

# --- VÒNG LẶP THỬ NGHIỆM VÀ QUÉT NGƯỠNG ---
for detector in DETECTORS:
    for model_name in MODELS:
        combo_name = f"{detector.upper()} + {model_name}"
        print(f"🔄 Đang trích xuất Vector cho: [{combo_name}]...")
        start_time = time.time()
        
        # A. Trích xuất CSDL
        registered_db = {}
        for file_path in db_files:
            filename = os.path.basename(file_path)
            prefix = filename.split('_')[0].lower()
            person_name = NAME_MAP.get(prefix, "Người lạ")
            if person_name == "Người lạ":
                continue
            try:
                res = DeepFace.represent(
                    img_path=file_path,
                    model_name=model_name,
                    detector_backend=detector,
                    enforce_detection=False
                )
                if len(res) > 0 and "embedding" in res[0]:
                    if person_name not in registered_db:
                        registered_db[person_name] = []
                    registered_db[person_name].append(res[0]["embedding"])
            except Exception:
                pass

        if not registered_db:
            print(f"  ❌ Detector '{detector}' không trích xuất được CSDL. Bỏ qua.\n")
            continue

        # B. Trích xuất & Tính khoảng cách nhỏ nhất cho từng ảnh Test (Tối ưu tốc độ)
        y_true = []
        min_distances = []
        best_matches = []
        failed_detections = 0

        for sample in test_samples:
            img_path = sample["image_path"]
            true_name = sample["true_name"]
            y_true.append(true_name)
            
            try:
                res = DeepFace.represent(
                    img_path=img_path,
                    model_name=model_name,
                    detector_backend=detector,
                    enforce_detection=False
                )
                
                if len(res) == 0:
                    failed_detections += 1
                    min_distances.append(float("inf"))
                    best_matches.append("Người lạ")
                    continue

                test_emb = res[0]["embedding"]
                min_dist = float("inf")
                match_person = "Người lạ"
                
                for person_name, embeddings in registered_db.items():
                    for db_emb in embeddings:
                        dist = cosine(test_emb, db_emb)
                        if dist < min_dist:
                            min_dist = dist
                            match_person = person_name
                
                min_distances.append(min_dist)
                best_matches.append(match_person)

            except Exception:
                failed_detections += 1
                min_distances.append(float("inf"))
                best_matches.append("Người lạ")

        # C. THRESHOLD SWEEPING: Quét dải ngưỡng để tìm ngưỡng tối ưu nhất
        best_combo_acc = -1.0
        best_combo_thresh = None
        best_combo_far = 0.0
        best_combo_frr = 0.0

        stranger_total = sum(1 for t in y_true if t == "Người lạ")
        family_total = sum(1 for t in y_true if t != "Người lạ")

        for th in THRESH_RANGE:
            y_pred_th = []
            for dist, match in zip(min_distances, best_matches):
                if dist <= th:
                    y_pred_th.append(match)
                else:
                    y_pred_th.append("Người lạ")

            acc = balanced_accuracy_score(y_true, y_pred_th) * 100
            
            # Chọn ngưỡng cho Acc cao nhất (hoặc ưu tiên FAR thấp nếu Acc bằng nhau)
            if acc > best_combo_acc:
                best_combo_acc = acc
                best_combo_thresh = round(th, 2)
                
                # Tính FAR, FRR tại ngưỡng này
                far_cnt = sum(1 for t, p in zip(y_true, y_pred_th) if t == "Người lạ" and p != "Người lạ")
                frr_cnt = sum(1 for t, p in zip(y_true, y_pred_th) if t != "Người lạ" and p != t)
                best_combo_far = round((far_cnt / stranger_total * 100), 2) if stranger_total > 0 else 0.0
                best_combo_frr = round((frr_cnt / family_total * 100), 2) if family_total > 0 else 0.0

        elapsed_time = round(time.time() - start_time, 2)
        
        final_summary.append({
            "Detector": detector,
            "Model": model_name,
            "Ngưỡng tối ưu": best_combo_thresh,
            "Balanced Acc (%)": round(best_combo_acc, 2),
            "FAR (%)": best_combo_far,
            "FRR (%)": best_combo_frr,
            "Lỗi Detect": failed_detections,
            "Thời gian (s)": elapsed_time
        })
        
        print(f"  🎯 Ngưỡng tối ưu: {best_combo_thresh} | Acc: {round(best_combo_acc, 2)}% | FAR: {best_combo_far}% | FRR: {best_combo_frr}%\n")

# --- ĐÁNH GIÁ VÀ XUẤT BÁO CÁO ---
df_summary = pd.DataFrame(final_summary)
df_summary = df_summary.sort_values(by=["Balanced Acc (%)", "FAR (%)"], ascending=[False, True])

print("=========================================================================================")
print("=== BÁO CÁO KẾT QUẢ TỐI ƯU: TÌM NGƯỠNG TỐT NHẤT CHO TỪNG TỔ HỢP ===")
print("=========================================================================================")
print(df_summary.to_string(index=False))

df_summary.to_csv(CSV_RESULT_PATH, index=False, encoding='utf-8-sig')
print(f"\n💾 Đã lưu kết quả tối ưu vào file '{CSV_RESULT_PATH}'!")


🚀 Bắt đầu quét dải ngưỡng cho 12 tổ hợp mô hình...

🔄 Đang trích xuất Vector cho: [RETINAFACE + Facenet]...
26-08-07 04:03:50 - retinaface.h5 will be downloaded from the url https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5
To: C:\Users\Tien Luu\.deepface\weights\retinaface.h5
100%|██████████| 119M/119M [00:04<00:00, 24.6MB/s] 
